# Using Numerical Optimisation to predict Vanilla Tables

## Imports

In [40]:
import pickle
from math import pi, sqrt, e, log
import functions as func
from scipy.optimize import minimize, Bounds
from importlib import reload
from statistics import NormalDist
reload(func)

<module 'functions' from 'c:\\Users\\psymf9\\PhD Repositories\\cherry-picking-reduction-functions\\plots\\higher_costs\\N_16\\constructing_table\\functions.py'>

## Constants

In [31]:
N = 2 ** 16
p = 1-e**-2 # our table coverage

# keep track of number of reductions so can take that away from cost at the end
num_reductions = 0

## Numerical Optimisation

In [32]:
# Optimise 
# add vcost to bound the optimisation
def optimise_kj(N, p, alpha, factor, lower_bound):   # add v_cost if want vanilla cost to be set beforehand
    t = round(log(1-p)/log(1-N**(-1/3))) # Calculate t
    mt_target = N**(2/3) # our target mt
    m_0 = round(mt_target/(1-alpha))    # our starting m_0
    
    ## set the bound and target function
    # TODO: change upper bound when needed
    bound = Bounds([lower_bound for i in range(t)], [1 for i in range(t)])
    
    # our target function that needs to be > 0
    # instead of using the target function, we can use the cost function as the objective function
    # need to get the cost of a vanilla rainbow table with the same parameters
    
    # Const * P(vanilla) - P(cherry) > 0
    v_cost, m = func.vanilla_cost(N, m_0, t)
    v_cost = v_cost * factor
    ineq_cons = {'type': 'ineq', 'fun' : lambda x: v_cost - func.cost(x, N, m_0)[0]}

    ## use 1 as starting values for cherry-picking
    starting_values = [1 for i in range(t)]
    
    ## call the optimizer
    # maximise the 
    # 5000000 normally
    res = minimize(lambda Kj: func.m_t(Kj, N, m_0), starting_values, bounds=bound, constraints=ineq_cons,method = "SLSQP", options={'disp': True, "maxiter": 5000000, "eps": 1, "ftol": 1})
    
    ## res.x is the result of the optimization
    final_cost, m_values = func.cost(res.x, N, m_0) # calculate the final cost and m values
    return res.x, m_0, final_cost, m_values

## Generate results

In [ ]:
result = optimise_kj(N, p, 0.95, 1, 1)
kjs, m_0, final_cost, m_values = result
# store in pickle file
with open("vanilla_N_16__alpha_0.95_sim_results.pkl", "wb") as f:
    pickle.dump(result, f)


In [36]:
final_cost

np.float64(404494.0249884667)

In [34]:
func.vanilla_cost(N, m_0, round(log(1-p)/log(1-N**(-1/3))))

(405354.5376762991,
 [32510,
  25629.764381344437,
  21212.39152285474,
  18121.80138209917,
  15832.231480093644,
  14065.074940120903,
  12658.286197965135,
  11510.939764085968,
  10556.781512869413,
  9750.460965659317,
  9059.860019656786,
  8461.578174346294,
  7938.153006889501,
  7476.283043523472,
  7065.654353924867,
  6698.143954625819,
  6367.265762936775,
  6067.776947630526,
  5795.392902461113,
  5546.5773530019505,
  5318.3854258166975,
  5108.344693499246,
  4914.3638735190325,
  4734.661949377332,
  4567.712568577794,
  4412.200003832222,
  4266.983962168633,
  4131.071232492774,
  4003.5926678976903,
  3883.7843657959675,
  3770.972177968717,
  3664.5588820261037,
  3564.0134950302963,
  3468.862322784291,
  3378.6814242033142,
  3293.090236180564,
  3211.746155441331,
  3134.33991370444,
  3060.591613729601,
  2990.2473185245326,
  2923.076105614011,
  2858.867513964491,
  2797.429323771583,
  2738.585619510166,
  2682.175094924256,
  2628.049565387715,
  2576.07265

In [39]:
len(kjs)

80

In [43]:
# try and calculate the last m_t
m = m_values[-1]
vcost = final_cost


# Calculate E1 and E2
E1 = (1 - 1 / N) ** m
E2 = (1 - 2 / N) ** m
average = N * (1 - E1)

# check variance
if N * ((N - 1) * E2 + E1 - N * E1 ** 2) < 0:
    # if it is negative (error), set it to 0
    variance = 0
else:
    # count the variance otherwise
    variance = sqrt(N * ((N - 1) * E2 + E1 - N * E1 ** 2))

vcost =  vcost + m + 1 / 577.44 

# find the m_j+1
m = average + NormalDist().inv_cdf((1 - pi / 8) / (1 - pi / 4 + 1)) * variance

print(m)
print(vcost)

1541.392961175683
406053.8237331421
